In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

def load_data():
    #读入数据
    test = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/test.csv')
    train = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/train.csv')
    sample_submission = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/sample_submission.csv')
    stores = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/stores.csv')
    holidays = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/holidays_events.csv')
    oil = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/oil.csv')
    transactions = pd.read_csv('/kaggle/input/store-sales-time-series-forecasting/transactions.csv')
    return train, test, stores, oil, transactions, holidays


In [ ]:
#数据探索
def explore_data(train,stores,oil):
#查看数据
    print("训练数据的形状：",train.shape)
    print("商店数据的形状:",stores.shape)
    print("油价数据的形状:",oil.shape)
    print("\n训练数据的基本信息:")
    print(train.info())#不会出现缺失值数量。只是会把该数据的基本情况体现出来
    print("\n前五行数据:")
    print(train.head())#默认是表示出前5行的数据如果需要10行的话，就写10就好

    print(train['date'].dtype)#输出结果为object

    #日期范围分析#pd.to_datetime(()将该列的数据类型转换一下方便计算和操
    train['date'] = pd.to_datetime(train['date'])
    print(train['date'].dtype)#此时输出的结果为datetime64[ns]
    print(f"\n日期范围:{train['date'].min()}到{train['date'].max()}")

#销售分布
    #figure创建图形对象
    #figsize=(15,10)设置图形大小宽15高10
    plt.figure(figsize=(15,10))

    #        第一个子图:销售分布直方图
    #选择4个位置中的第一个位置,subplot(x,y,z)中x为总行数,y为总列数,z为当前所处的子图位置
    plt.subplot(2,2,1)
    train['sales'].hist(bins = 50)#挑选出sales的数据列,来进行绘制直行图,分为50个区间
    plt.title('销售分布')
    plt.xlabel('销售额')
    plt.ylabel('频次')

    #        第二个子图:商店总销售额
    plt.subplot(2,2,2)
    #将数据先是按照store_nbr即商店编号进行分组，然后在分组的前提下选择sales列的数据来计算
    #.sort_values(ascending = False)对进行计算的数据进行排序;ascending = False进行降序排列
    store_sales = train.groupby('store_nbr')['sales'].sum().sort_values(ascending = False)
    #将销售量最高的前10个绘制成柱形图
    store_sales.head(10).plot(kind='bar')
    plt.title('Top 10商店的总销售额')
    #xticks表示对x轴刻度进行旋转;rotation = 45即旋转角度为45°
    plt.xticks(rotation = 45)

    #        第三个子图:产品类别
    plt.subplot(2,2,3)
    family_sales = train.groupby('family')['sales'].sum().sort_values(ascending = False)
    family_sales.head(10).plot(kind='bar')
    plt.title('Top 10产品类别销售额')
    plt.xticks(rotation = 45)

    #        第四个子图:
    plt.subplot(2,2,4)
    #时间序列趋势
    daily_sales = train.groupby('date')['sales'].sum()
    #对结果进行绘制折线图
    daily_sales.plot()
    plt.title('每日总销售额趋势')
    plt.xlabel('日期')
    plt.ylabel('销售额')
    #plt.tight_layout()自动调整子图之间的间距，防止重叠
    plt.tight_layout()
    plt.show()#显示并关闭第一个图形
    
    return daily_sales


In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
#时间序列季节分析

def seasonal_analysis(daily_sales):
    #进行重采样,以周和月为频率
    weekly_sales = daily_sales.resample('W').mean()
    monthly_sales = daily_sales.resample('M').mean()

    #在不同时间下的尺度中销售趋势对比(时间长度不同)
    plt.figure(figsize = (15,10))
    plt.subplot(3,1,1)
    daily_sales.plot()
    plt.title('日销售额')

    plt.subplot(3,1,2)
    weekly_sales.plot()
    plt.title('周销售额')

    plt.subplot(3,1,3)
    monthly_sales.plot()
    plt.title('月销售额')

    plt.tight_layout()
    plt.show()

    #季节性分解
    decomposition = seasonal_decompose(daily_sales,period = 365,model = 'additive')
    plt.figure(figsize = (15,12))
    decomposition.plot()#自动创建4个子图
    plt.show()


In [ ]:
from sklearn.preprocessing import LabelEncoder

#创建时间序列的特征
#df = train.copy()
#print("初始df形状：",df.shape)
def create_features(df,oil,stores,holidays,transactions):
    #合并商店信息
    df['date']=pd.to_datetime(df['date'])
    df = df.merge(stores,on='store_nbr',how='left')
    print("合并商店之后的df形状：",df.shape)
    #合并油价信息
    oil['date'] = pd.to_datetime(oil['date'])
    df['date'] = pd.to_datetime(df['date'])
    df = df.merge(oil,on='date',how='left')

    #处理油价缺失值
    #旧版写法：df['dcoilwtico'] = df['dcoilwtico'].fillna(method='ffill')
    #最新写法
    df['dcoilwtico'] = df['dcoilwtico'].ffill()

    #创建时间特征
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month
    df['day'] = df['date'].dt.day
    df['dayofweek'] = df['date'].dt.dayofweek
    df['dayofyear'] = df['date'].dt.dayofyear
    df['weekofyear'] = df['date'].dt.isocalendar().week
    df['quarter'] = df['date'].dt.quarter
    df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
    df['is_month_end'] = df['date'].dt.is_month_end.astype(int)
    df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(int)

    #创建滞后特征
    for lag in [1,7,14,28]:
        df[f'sales_lag_{lag}'] = df.groupby(['store_nbr','family'])['sales'].shift(lag)
    
    #创建滚动统计特征
    for window in [7,14,28]:
        df[f'sales_roll_mean_{window}'] = df.groupby(['store_nbr','family'])['sales'].transform(
            lambda x:x.rolling(window,min_periods=1).mean())
        df[f'sales_roll_std_{window}'] = df.groupby(['store_nbr','family'])['sales'].transform(
            lambda x:x.rolling(window,min_periods=1).std())
    
    
    #处理假期数据
    holidays['date'] = pd.to_datetime(holidays['date'])
    holidays = holidays[holidays['transferred'] == False]
    
    #创建假期特征
    holiday_dates = holidays['date'].unique()
    df['is_holiday'] = df['date'].isin(holiday_dates).astype(int)
    
    #合并交易数据
    transactions['date'] = pd.to_datetime(transactions['date'])
    df = df.merge(transactions,on=['date','store_nbr'],how='left')
    
    #编码分类变量
    le_family = LabelEncoder()
    le_city = LabelEncoder()
    le_state = LabelEncoder()
    le_type = LabelEncoder()
    df['family_encoded'] = le_family.fit_transform(df['family'])
    df['city_encoded'] = le_city.fit_transform(df['city'])
    df['state_encoded'] = le_state.fit_transform(df['state'])
    df['type_encoded'] = le_type.fit_transform(df['type'])
    
    #创建交互特征
    df['store_family'] = df['store_nbr'].astype(str)+'_'+df['family']
    le_store_family = LabelEncoder()
    df['store_family_encoded'] = le_store_family.fit_transform(df['store_family'])
    
    #return df
    
    #准备滞后特征
    for lag in lags:
        df[f'{target_col}_lag_{lag}'] = df.groupby(['store_nbr','family'])[target_col].shift(lag)

    return df


In [ ]:
#模型训练

#训练多个模型并进行比较
def train_models(X_train,y_train,X_val,y_val):
    models = {}

    #XGBoost模型
    xgb_model = xgb.XGBRegressor(
        n_estimators = 1000,
        learning_rate = 0.1,
        max_depth = 8,
        subsample = 0.8,
        colsample_bytree = 0.8,
        random_state = 42
    )
    xgb_model.fit(
        X_train,y_train,
        eval_set = [(X_val,y_val)],
        early_stopping_rounds = 50,
        verbose = False
    )
    model['xgb'] = xgb_model

    #LightGBM模型
    lgb_model = lgb.LGBMRegressor(
        n_estimators = 1000,
        learning_rate = 0.1,
        max_depth = 8,
        subsample = 0.8,
        colsample_bytree = 0.8,
        random_state = 42
    )
    lgb_model.fit(
        X_train,y_train,
        eval_set = [(X_val,y_val)],
        early_stopping_rounds = 50,
        verbose = False
    )
    model['lgb'] = lgb_model
    return models
    
#评估模型性能,X_val是验证集的特征数据,y_val是验证集的真实目标值
def evaluate_models(models,X_val,y_val):
    results = {}

    for name,model in models.items():
        y_pred = model.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val,y_pred))
        results[name] = rmse
        print(f"{name.upper()} RMSE:{rmse:.4f}")
    return results


#绘制特征重要性图
def plot_feature_importance(model,feature_names,top_n = 20):
    importance = model.feature_importances_
    indices = np.argsort(importance)[::-1]
    plt.figure(figsize = (10,8))
    plt.title("Feature Importance")
    plt.barh(range(top_n),importance[indices[:top_n]][::-1])
    plt.yticks(range(top_n),[feature_names[i] for i in indices[:top_n]][::-1])
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
    

In [ ]:
#完整训练流程

#主训练流程
def main():
    print("加载数据")
    train,test,stores,oil,transactions,holidays = load_data()
    print("探索性数据分析")
    daily_sales = explore_data(train,stores,oil)
    seasonal_analysis(daily_sales)
    print("特征工程")
    train_sample = train.sample(frac = 0.1,random_state = 42)#使用10%的数据作为示例
    df_features = create_features(train_sample,oil,stores,holidays,transactions)
    #准备滞后特征
    df_features = prepare_lagged_features(df_features)
    #删除包含NaN的行
    df_features = df_features.dropna()
    #选择特征列
    feature_columns = [
        'store_nbr', 'family_encoded', 'city_encoded', 'state_encoded', 'type_encoded',
        'dcoilwtico', 'year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear',
        'quarter', 'is_month_start', 'is_month_end', 'is_weekend', 'is_holiday',
        'transactions', 'sales_lag_1', 'sales_lag_7', 'sales_lag_14', 'sales_lag_28',
        'sales_roll_mean_7', 'sales_roll_mean_14', 'sales_roll_mean_28',
        'sales_roll_std_7', 'sales_roll_std_14', 'sales_roll_std_28',
        'store_family_encoded'
    ]
    X = df_features[feature_columns]
    y = df_features['sales']

    print("划分训练集和验证集")
    X_train,X_val,y_train,y_val = train_test_split(X,y,test_size = 0.2,random_state = 42,shuffle = False)

    print("训练模型")
    models = train_model(X_train,y_train,X_val,y_val)

    print("模型评估")
    results = evaluate_models(models,X_val,y_val)

    print("特征重要性分析")
    best_model = models['xgb']
    plot_feature_importance(best_model,feature_columns)
    
    return models,feature_columns

    

In [ ]:
#预测和提交

#创建提交文件
def create_submission(models,test,stores,oil,holidays,transactions,feature_colimns):
    print("准备测试数据")
    test_processed = create_features(test,oil,stores,holidays,transactions)
    for col in feature_columns:
        if col not in test_processed.columns:
            test_processed[col] = 0
    #选择最佳模型进行预测
    best_model = models['xgb']

    print("进行预测")
    X_test = test_processed[feature_columns]
    predictions = best_model.predict(X_test)

    #创建提交文件
    submission = test[['id']].copy()
    submission['sales'] = predictions
    
    #确保销售额为非负
    submission['sales'] = submission['sales'].clip(lower = 0)
    #保存提交文件
    submission.to_csv('submission.csv',index = False)
    print("提交文件已保存为'submission.csv")
    
    return submission
    